# Frameworks de Big Data de Última Generación (2025) + Comparativo de Desempeño

**Objetivo:** Explorar el ecosistema moderno (Spark/Flink/Kafka/NoSQL/Vector DBs/DuckDB/Polars) y ejecutar un comparativo reproducible entre un enfoque clásico y uno moderno en un problema de escala.

---
## 1. Panorama 2025: Marco de Referencia

| Capa | Tecnologías Clave | Casos de uso |
|------|--------------------|--------------|
| Ingesta/Eventos | Kafka, Pulsar, Event Hubs | Telemetría, logs, CDC, IoT |
| Stream Proc. | Flink 1.18+, Spark Structured Streaming | Ventanas, estado, joins en tiempo real |
| Batch Proc. | Spark 3.5+, dbt Core, Ray | Transformaciones, ML feature pipelines |
| Query Serving | DuckDB, Trino, BigQuery, Snowflake | SQL interactivo, BI |
| NoSQL | Cassandra, DynamoDB, MongoDB | Clave-valor, wide-column, documentos |
| Vector DB | Milvus, pgvector, Pinecone | Búsqueda semántica, RAG |
| ML/FS | Feast/Tecton, MLflow | Feature store, model registry |
| Formatos | Parquet, Delta, Iceberg, Hudi | Columnares, ACID, time travel |

---
## 2. Spark vs Flink (visión)

| Aspecto | Spark 3.5+ | Flink 1.18+ |
|---------|------------|-------------|
| Modelo | Micro-batch/continuous | Streaming nativo (event-time, estado) |
| Latencia | Baja (ms–s) | Muy baja (ms) |
| Batch | Excelente | Bueno (Blink planner) |
| APIs | DataFrame, SQL | DataStream, Table/SQL |
| ETL | Muy fuerte (ecosistema) | Fuerte (stateful streaming) |

---
## 3. NoSQL y Bases Vectoriales

- Cassandra/DynamoDB: wide-column/kv para escrituras masivas y latencias estables.
- MongoDB/DocumentDB: flexibilidad de esquema para documentos JSON.
- Vector DBs (Milvus, pgvector, Pinecone): búsquedas por similitud en embeddings (RAG/LLMs).

---
## 4. Comparativo de Desempeño: Clásico vs Moderno

Evaluaremos una tarea típica: agregaciones y join sobre datos sintéticos (~5–10M filas).

- Clásico: pandas (memoria única, single-thread).
- Moderno: Polars (vectorizado, multi-thread) y/o DuckDB (vectorizado + SQL in-process).

Notas:
- Si no están instalados `polars` o `duckdb`, el notebook seguirá con pandas y mostrará advertencias.
- Este comparativo es ilustrativo; en entornos reales use datasets mayores y hardware adecuado.

<!-- course-nav -->
> Ruta: [Inicio del repositorio](../README.md) | [Índice del nivel](README.md)  
> Anterior: [Arquitectura AI-Ready: Feature Stores y Streaming para ML](15_arquitectura_ai_ready_feature_stores_streaming.ipynb) | Siguiente: [Mini Data Mesh: Dos Dominios con Catálogo Compartido](17_mini_data_mesh_dos_dominios_catalogo_compartido.ipynb)


<!-- course-quality-structure -->
## Prerrequisitos

- Haber revisado los notebooks anteriores de la ruta o contar con experiencia equivalente.
- Conocer los conceptos básicos mencionados en el README del nivel.
- Tener disponible el repositorio local y las dependencias de `requirements.txt` cuando se ejecuten celdas de código.

## Preparación

1. Trabaja desde la raíz del repositorio.
2. Activa tu entorno de Python.
3. Instala dependencias con `pip install -r requirements.txt` si aún no lo hiciste.
4. Ejecuta las celdas en orden para conservar el contexto de variables y resultados.

## Mapa del notebook

1. 1. Panorama 2025: Marco de Referencia
2. 2. Spark vs Flink (visión)
3. 3. NoSQL y Bases Vectoriales
4. 4. Comparativo de Desempeño: Clásico vs Moderno
5. 5. Interpretación de Resultados
6. 6. Guía de Selección (rápida)
7. 7. Buenas Prácticas 2025

## Diagrama de continuidad

```mermaid
flowchart LR
    A["Notebook anterior"] --> B["Frameworks de Big Data de Última Generación (2025) + Comparativo de Desempeño"] --> C["Notebook siguiente"]
```


In [ ]:
import time
import random
import pandas as pd

try:
    import polars as pl
    HAS_POLARS = True
except Exception:
    HAS_POLARS = False

try:
    import duckdb
    HAS_DUCKDB = True
except Exception:
    HAS_DUCKDB = False

N = 100_000
random.seed(42)
users = pd.DataFrame({
    'user_id': range(N),
    'country': [random.choice(['US', 'ES', 'MX', 'BR', 'DE', 'FR']) for _ in range(N)],
    'age': [random.randint(18, 70) for _ in range(N)]
})
events = pd.DataFrame({
    'user_id': [random.randint(0, N - 1) for _ in range(N)],
    'event_type': [random.choice(['view', 'click', 'purchase']) for _ in range(N)],
    'amount': [random.random() * 100 for _ in range(N)]
})

def benchmark_pandas(users, events):
    t0 = time.time()
    joined = events.merge(users, on='user_id', how='left')
    result = joined.groupby(['country', 'event_type'])['amount'].sum().reset_index()
    return time.time() - t0, result.head()

def benchmark_polars(users, events):
    if not HAS_POLARS:
        return None, None
    t0 = time.time()
    result = (
        pl.from_pandas(events)
        .join(pl.from_pandas(users), on='user_id', how='left')
        .group_by(['country', 'event_type'])
        .agg(pl.col('amount').sum())
    )
    return time.time() - t0, result.head()

def benchmark_duckdb(users, events):
    if not HAS_DUCKDB:
        return None, None
    t0 = time.time()
    con = duckdb.connect(':memory:')
    con.register('users', users)
    con.register('events', events)
    query = '''
        SELECT u.country, e.event_type, SUM(e.amount) AS amount
        FROM events e
        LEFT JOIN users u USING (user_id)
        GROUP BY u.country, e.event_type
        ORDER BY u.country, e.event_type
    '''
    result = con.execute(query).fetchdf()
    con.close()
    return time.time() - t0, result.head()

print('=== Benchmarks locales (tiempos en segundos) ===')
p_dt, p_head = benchmark_pandas(users, events)
print(f'pandas: {p_dt:.3f}s')
if HAS_POLARS:
    pl_dt, _ = benchmark_polars(users, events)
    print(f'polars: {pl_dt:.3f}s')
else:
    print('polars: no instalado; benchmark opcional omitido')
if HAS_DUCKDB:
    dd_dt, _ = benchmark_duckdb(users, events)
    print(f'duckdb: {dd_dt:.3f}s')
else:
    print('duckdb: no instalado; benchmark opcional omitido')
print('\nMuestra de salida pandas:')
print(p_head)


## 5. Interpretación de Resultados

- Polars y DuckDB suelen superar a pandas en operaciones de agregación y joins pesados gracias a ejecución vectorizada y paralelismo.
- En producción, Spark/Flink se usan para datasets distribuidos (TB–PB) y cargas continuas.
- DuckDB y Polars complementan para análisis locales rápidos (exploración, prototipos, CI).

## 6. Guía de Selección (rápida)

- ¿Datos en PB + múltiples equipos? → Spark/Flink + formatos abiertos (Iceberg/Delta).
- ¿SQL local interactivo en archivos Parquet? → DuckDB.
- ¿Transformaciones locales rápidas en DataFrames? → Polars.
- ¿Búsqueda semántica/LLMs? → Vector DB (pgvector/Milvus).

## 7. Buenas Prácticas 2025

- Mantener datos en formatos abiertos (evitar lock-in).
- Definir contrato de esquemas y estrategia de evolución.
- Monitorear costo/latencia con SLOs y auto-escalado.
- Integrar catálogo + linaje + calidad (OpenLineage/Atlas/DataHub).

<!-- course-quality-close -->
## Ejercicio de cierre

Revisa el caso trabajado en el notebook y documenta una decisión de arquitectura: contexto, opción elegida, alternativas descartadas, riesgos y métricas de validación.

## Evaluación

- El notebook se ejecuta en orden sin depender de estado oculto.
- Las decisiones técnicas están justificadas con trade-offs claros.
- Los resultados, diagramas o tablas apoyan la explicación y no reemplazan el razonamiento.
- Las referencias usadas son oficiales, primarias o están incluidas en la bibliografía del curso.

## Referencias

- Apache Parquet: https://parquet.apache.org/
- Referencias del curso: ../04-recursos/referencias.md
